In [17]:
import numpy as np
from typing import List, Dict, Any
from bertopic import BERTopic
import bertopic
from hdbscan import HDBSCAN
from umap import UMAP
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import traceback
import pandas as pd

import sys
import os
# Adds the parent directory ('..') to the Python path
sys.path.append(os.path.abspath(os.path.join("..")))
from app.models.topic_result import TopicResult


In [ ]:
def get_default_params() -> Dict[str, Any]:
    return {
        'bertopic_min_topic_size': 15,
        'bertopic_embedding_model': 'all-MiniLM-L6-v2',
        'bertopic_calculate_probabilities': True,
        'umap_n_neighbors': 15,
        'umap_n_components': 2,
        'umap_metric': 'cosine',
        'umap_low_memory': False,
        'hdbscan_min_samples': 10,
        'hdbscan_metric': 'euclidean'
    }

In [ ]:
def train(texts: List[str], **params) -> TopicResult:
    """
    Train BERTopic model and return a standardized TopicResult.
    """
    try:
        # 1. Configure UMAP
        umap_model = UMAP(
            n_neighbors=int(params.get('umap_n_neighbors', 15)),
            n_components=int(params.get('umap_n_components', 5)),
            metric=params.get('umap_metric', 'cosine'),
            low_memory=bool(params.get('umap_low_memory', False)),
            random_state=8
        )

        # 2. Configure HDBSCAN
        clusterer = HDBSCAN(
            min_cluster_size=int(params.get('bertopic_min_topic_size', 5)),
            min_samples=int(params.get('hdbscan_min_samples', 5)),
            metric=params.get('hdbscan_metric', 'euclidean'),
            prediction_data=True
        )

        # 3. Configure Vectorizer and CT-IDF
        vectorizer = CountVectorizer(stop_words="english")
        ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

        # 4. Initialize BERTopic
        model = BERTopic(
            min_topic_size=int(params.get('bertopic_min_topic_size', 10)),
            embedding_model=params.get('bertopic_embedding_model', 'all-MiniLM-L6-v2'),
            calculate_probabilities=bool(params.get('bertopic_calculate_probabilities', True)),
            umap_model=umap_model,
            hdbscan_model=clusterer,
            vectorizer_model=vectorizer,
            ctfidf_model=ctfidf_model
        )

        # Fit the model
        topics, probs = model.fit_transform(texts)

        # 1. Extract Topics (term -> weight)
        # BERTopic stores topics as (word, weight) tuples in get_topic()
        topics_list = []
        # Topic -1 is outliers, we usually exclude it from the top topics list
        for topic_id in sorted(model.get_topics().keys()):
            if topic_id == -1:
                continue
            topic_data = model.get_topic(topic_id)
            # Convert [(word, weight), ...] to {word: weight, ...}
            topics_list.append({word: weight for word, weight in topic_data})

        # 2. Extract Doc-Topic assignments
        # If calculate_probabilities=True, probs is a numpy array [n_docs, n_topics]
        doc_topics = []
        if isinstance(probs, np.ndarray):
            # Ensure probs is 2D even if only one topic was found
            if probs.ndim == 1:
                probs = probs.reshape(-1, 1)

            # Map each doc to a dict of {topic_id: probability}
            # BERTopic's probs array corresponds to the topics in model.get_topics()
            # Topic -1 is not usually in the probs array
            topic_names = list(model.get_topics().keys())
            # Filter out -1 for probability mapping
            filtered_topics = [t for t in topic_names if t != -1]

            for doc_prob in probs:
                # Map prob index to actual topic_id
                doc_map = {filtered_topics[i]: prob for i, prob in enumerate(doc_prob)}
                doc_topics.append(doc_map)
        else:
            # Fallback to hard assignment
            for t in topics:
                doc_topics.append({t: 1.0})

        # 3. Metadata & Model Info
        metadata = {
            'model': model,
            'texts': texts,
            'num_topics': len(model.get_topics()) - 1 # exclude -1
        }

        model_info = {
            'algorithm': 'BERTopic',
            'version': bertopic.__version__,
            'params': str(params),
            'num_documents': str(len(texts)),
            'num_topics': str(len(model.get_topics()) - 1)
        }

        return TopicResult(
            topics=topics_list,
            doc_topics=doc_topics,
            metadata=metadata,
            model_info=model_info
        )

    except Exception as e:
        call_stack = traceback.format_exc()
        raise RuntimeError(f"BERTopic training failed: {str(e)}\n\n {call_stack}") from e

In [ ]:
df = pd.read_csv("../sample_dataset copy.csv")
params = get_default_params()
result = train(texts=df["Abstract"], **params)

In [21]:
topic_model = result.metadata["model"]

In [22]:
print(result.metadata["num_topics"])

22


In [23]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,250,-1_sector_digital_political_services,"[sector, digital, political, services, artific...",[Purpose: The purpose of this paper is to expl...
1,0,247,0_traffic_learning_smart_proposed,"[traffic, learning, smart, proposed, network, ...",[Background. The environment has been signific...
2,1,58,1_algorithmic_algorithms_criminal_fairness,"[algorithmic, algorithms, criminal, fairness, ...","[As is common across the public sector, the UK..."
3,2,34,2_administration_psd_sdgs_sector,"[administration, psd, sdgs, sector, digital, p...",[The article reflects research on the dynamics...
4,3,24,3_perceived_adoption_trust_tam,"[perceived, adoption, trust, tam, employees, a...",[BACKGROUND AND OBJECTIVES: Urban administrato...
5,4,21,4_sovereignty_european_surveillance_rights,"[sovereignty, european, surveillance, rights, ...",[Artificial intelligence (AI) and machine lear...
6,5,20,5_green_environmental_japanese_energy,"[green, environmental, japanese, energy, city,...",[Smart cities play a vital role in the growth ...
7,6,17,6_automated_administrative_law_rule,"[automated, administrative, law, rule, legal, ...",[Public sector digitalisation is transforming ...
8,7,17,7_sentiment_opinion_tweets_arabic,"[sentiment, opinion, tweets, arabic, twitter, ...",[Social media materialized as an influential p...
9,8,16,8_administrative_judicial_legal_judiciary,"[administrative, judicial, legal, judiciary, b...",[The European Union has taken a leading role i...


In [24]:
topic_model.visualize_topics()